# RSNA 2018 — Pipeline de Pré-processamento e Extração de Características
Este notebook prepara os dados do experimento: baixa o conjunto RSNA, executa o pré-processamento DICOM, gera regiões candidatas e extrai HOG/LBP/GLCM.
As saídas alimentam diretamente o pipeline de modelagem, avaliação e detecção.

### 1. Repositório e dependências
Clona a branch com o código (mesmo código do repositório, sem cópia solta). Ajuste `BRANCH` se o nome for outro.

In [ ]:
BRANCH = "main"
!git clone -q -b {BRANCH} https://github.com/erik-hp/ap1-rsna-pneumonia-baseline.git repo
%cd repo
!pip -q install -r requirements.txt
!pip -q install -e .
!pip -q install kaggle

### 2. Credenciais do Kaggle
Kaggle → Settings → API → *Generate New Token* (ou use o `kaggle.json` antigo na alternativa comentada).

In [ ]:
import os, getpass
os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Kaggle API token: ")

# Alternativa (kaggle.json):
# from google.colab import files; files.upload()
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

### 3. Download e extração (só treino estágio 2)
Se o nome do .zip ou da competição for diferente, copie o comando da aba *Data* do desafio.

In [ ]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p data
!mkdir -p data/raw && unzip -q data/rsna-pneumonia-detection-challenge.zip "stage_2_train_images/*" stage_2_train_labels.csv -d data/raw
!ls data/raw/stage_2_train_images | wc -l

### 4. Testes rápidos do pipeline

In [ ]:
!pytest -q tests/test_features.py

### 5. Extração da amostra final (seed 42, amostragem natural estratificada)
A execução científica final usa **500 pacientes**. O script registra os pacientes selecionados em `sample_manifest.csv`, permitindo auditar e reproduzir a amostra usada no artigo.


In [ ]:
!python scripts/extract_features.py --raw data/raw --out data/processed --sampling natural --n-total 500 --jobs -1


### 6. Validação da amostra e das matrizes antes de modelar


In [ ]:
import pandas as pd, numpy as np

sample = pd.read_csv("data/processed/sample_manifest.csv")
counts = sample["Target"].value_counts().sort_index().to_dict()
assert len(sample) == 500, f"Esperados 500 pacientes, encontrados {len(sample)}"
assert sample["patientId"].nunique() == 500, "A amostra deve conter 500 pacientes únicos"
assert counts == {0: 387, 1: 113}, f"Distribuição inesperada: {counts}"
print("sample_manifest.csv", sample.shape, counts, "seed=42")

for f in ["features_all", "features_regions"]:
    d = pd.read_csv(f"data/processed/{f}.csv")
    X = d.drop(columns=[c for c in ["patientId", "Target", "x", "y", "width", "height", "max_iou"] if c in d])
    assert np.isfinite(X.values).all() and d.patientId.notna().all() and set(d.Target.unique()) <= {0, 1}
    assert {c.split("_")[0] for c in X.columns} <= {"hog", "lbp", "glcm"}
    print(f, d.shape, d.Target.value_counts().to_dict())


### 7. Salvar no Google Drive no mesmo caminho usado pelo notebook de modelagem


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PROCESSED = "/content/drive/MyDrive/RSNA_TP1/data/processed"
!mkdir -p "{DRIVE_PROCESSED}" && cp data/processed/*.csv "{DRIVE_PROCESSED}/"
!ls -lh "{DRIVE_PROCESSED}"
